# Task 2: Contracts, nested data and a star schema
CSEN1095 Data Engineering · W26

Pipeline: **validate** every line against a Pydantic contract → **quarantine** what fails (and re-sent duplicates) →
**flatten** the valid documents into `bookings` and `booking_tickets` → **model** a star schema in DuckDB → **store** it as Parquet in `star/`.

Every number shown below is computed from the file in `DATA`; nothing is typed in.

In [1]:
DATA = "bookings.jsonl"

In [2]:
import json
import math
import shutil
import datetime as dt
from collections import Counter
from pathlib import Path
from typing import Annotated, Literal, Optional

import duckdb
import pandas as pd
from pydantic import (BaseModel, ConfigDict, Field, StringConstraints,
                      ValidationError, field_validator, model_validator)

QUARANTINE_FILE = Path("quarantine.jsonl")
STAR_DIR = Path("star")

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 90)

## Part A: The contract (Lab 4)

### A.1 The contract as Pydantic models
One model per nested object (`Ticket`, `Student`, `Event`) and one for the whole `Booking` that uses the other three.

| Rule | Where it is enforced |
|---|---|
| `student.id` two digits, a dash, four digits | `StudentId` pattern `^[0-9]{2}-[0-9]{4}$` |
| `student.faculty` in MET / IET / EMS / Pharmacy / Management | `Literal` |
| `event` present with all five fields, `club` in the five clubs | `event: Event` is required (no default, `None` is rejected); all five `Event` fields required |
| at least one ticket | `Field(min_length=1)` |
| ticket `type` / `qty ≥ 1` / `price ≥ 0` | `Literal`, `Field(ge=1)`, `Field(ge=0)` |
| `payment`, `status` in the allowed values | `Literal` |
| `total` = Σ qty × price **(cross-field)** | `model_validator` |
| `booked_at` before the event's date **(cross-field)** | `model_validator` |

`booked_at` is normalised to UTC. "Before the event's date" is read literally: the booking must be strictly earlier than
00:00 UTC on the event's date. Money is compared with a tolerance of half a piastre (0.005 EGP) so float rounding never quarantines a correct booking.

In [3]:
StudentId = Annotated[str, StringConstraints(pattern=r"^[0-9]{2}-[0-9]{4}$")]
NonEmptyStr = Annotated[str, StringConstraints(min_length=1)]

# Fixed reason texts for the two cross-field rules, so they can be counted.
TOTAL_MISMATCH = "total does not equal the sum of qty x price over the tickets"
BOOKED_AFTER_EVENT = "booked_at is not before the event's date"


class Ticket(BaseModel):
    model_config = ConfigDict(extra="allow")  # keep drifted nested fields too

    type: Literal["standard", "vip", "student"]
    qty: int = Field(ge=1)
    price: float = Field(ge=0)


class Student(BaseModel):
    model_config = ConfigDict(extra="allow")  # keep drifted nested fields too

    id: StudentId
    name: NonEmptyStr
    faculty: Literal["MET", "IET", "EMS", "Pharmacy", "Management"]


class Event(BaseModel):
    model_config = ConfigDict(extra="allow")  # keep drifted nested fields too

    id: NonEmptyStr
    title: NonEmptyStr
    club: Literal["Robotics", "Debate", "Music", "Football", "Chess"]
    date: dt.date
    venue: NonEmptyStr


class Booking(BaseModel):
    # Unknown top-level fields are kept (not rejected): schema drift must not break the pipeline.
    model_config = ConfigDict(extra="allow")

    booking_id: NonEmptyStr
    booked_at: dt.datetime
    student: Student
    event: Event
    tickets: list[Ticket] = Field(min_length=1)
    total: float
    payment: Literal["card", "cash", "wallet"]
    status: Literal["confirmed", "cancelled"]
    promo_code: Optional[str] = None  # the drifted field (Part B.3): absent -> None

    @field_validator("booked_at")
    @classmethod
    def to_utc(cls, v: dt.datetime) -> dt.datetime:
        # booked_at is documented as UTC; a timestamp without an offset is taken as UTC.
        return v.replace(tzinfo=dt.timezone.utc) if v.tzinfo is None else v.astimezone(dt.timezone.utc)

    @model_validator(mode="after")
    def cross_field_rules(self) -> "Booking":
        problems = []
        expected = sum(t.qty * t.price for t in self.tickets)
        if not math.isclose(self.total, expected, abs_tol=0.005):
            problems.append(TOTAL_MISMATCH)
        event_starts = dt.datetime.combine(self.event.date, dt.time.min, tzinfo=dt.timezone.utc)
        if self.booked_at >= event_starts:
            problems.append(BOOKED_AFTER_EVENT)
        if problems:
            raise ValueError("; ".join(problems))
        return self

### A.2 + A.3 Validate every line, then drop re-sent duplicates
Each line is parsed and validated. A line that fails goes to the quarantine with its `booking_id`, its line number and the
reason, and is never used again. Only **valid** bookings take part in the duplicate check: the first valid copy of each
`booking_id` is kept, later copies are quarantined as `duplicate booking_id`.

The reason is built from Pydantic's errors as `field: message`; list positions are dropped (`tickets.type`, not `tickets.0.type`)
so the same rule always produces the same reason and can be counted. A `null` or absent field is reported as `missing`.

In [4]:
DUPLICATE = "duplicate booking_id"


def reason_from(err: ValidationError) -> str:
    """Turn a ValidationError into a stable, countable reason string."""
    parts = []
    for e in err.errors():
        path = ".".join(str(p) for p in e["loc"] if not isinstance(p, int))
        if e["type"] == "missing" or (path and e.get("input", "") is None):
            msg = "missing"
        else:
            msg = e["msg"].removeprefix("Value error, ")
        parts.append(f"{path}: {msg}" if path else msg)
    return "; ".join(dict.fromkeys(parts))  # unique, in order


def read_lines(path):
    """Yield (line_no, parsed document or None, error reason or None, raw line)."""
    with open(path, encoding="utf-8") as f:
        for line_no, raw in enumerate(f, start=1):
            raw = raw.rstrip("\r\n")
            if not raw.strip():
                yield line_no, None, "empty line", raw
                continue
            try:
                yield line_no, json.loads(raw), None, raw
            except json.JSONDecodeError as e:
                yield line_no, None, f"invalid JSON: {e.msg}", raw


lines_read = 0
docs = []         # every parsed document (used for the drift check in Part B)
valid = []        # Booking objects that passed the contract and are not duplicates
quarantine = []   # one dict per rejected line
seen_ids = set()

for line_no, doc, parse_error, raw in read_lines(DATA):
    lines_read += 1
    if parse_error:
        quarantine.append({"line": line_no, "booking_id": None, "reason": parse_error, "record": raw})
        continue
    docs.append((line_no, doc))
    booking_id = doc.get("booking_id") if isinstance(doc, dict) else None
    try:
        booking = Booking.model_validate(doc)                     # 1) the contract
    except ValidationError as err:
        quarantine.append({"line": line_no, "booking_id": booking_id,
                           "reason": reason_from(err), "record": doc})
        continue
    if booking.booking_id in seen_ids:                            # 2) re-sent copies
        quarantine.append({"line": line_no, "booking_id": booking_id,
                           "reason": DUPLICATE, "record": doc})
        continue
    seen_ids.add(booking.booking_id)
    valid.append(booking)

with open(QUARANTINE_FILE, "w", encoding="utf-8") as f:
    for q in quarantine:
        f.write(json.dumps(q, ensure_ascii=False) + "\n")

quarantine_df = pd.DataFrame(quarantine, columns=["line", "booking_id", "reason", "record"])
print(f"Saved {len(quarantine)} quarantined lines to {QUARANTINE_FILE}")
quarantine_df[["line", "booking_id", "reason"]]

Saved 24 quarantined lines to quarantine.jsonl


,line,booking_id,reason
0,11,B0011,"tickets.type: Input should be 'standard', 'vip' or 'student'"
1,16,B0016,tickets.qty: Input should be greater than or equal to 1
2,24,B0024,student.id: String should match pattern '^[0-9]{2}-[0-9]{4}$'
3,44,B0035,duplicate booking_id
4,50,B0049,event: missing
5,58,B0057,student.id: String should match pattern '^[0-9]{2}-[0-9]{4}$'
6,61,B0060,student.id: String should match pattern '^[0-9]{2}-[0-9]{4}$'
7,91,B0090,"tickets.type: Input should be 'standard', 'vip' or 'student'"
8,102,B0101,total does not equal the sum of qty x price over the tickets
9,107,B0106,student.id: String should match pattern '^[0-9]{2}-[0-9]{4}$'


### A.4 Report

In [5]:
passed = len(valid)
quarantined = len(quarantine)
assert lines_read == passed + quarantined, "every line must be either kept or quarantined"

print(f"Lines read:   {lines_read}")
print(f"Passed:       {passed}")
print(f"Quarantined:  {quarantined}\n")

reason_counts = (quarantine_df["reason"].value_counts()
                 .rename_axis("reason").reset_index(name="bookings"))
print("Quarantined per reason (one row per booking):")
display(reason_counts)

# A booking can break several rules at once; this breaks combined reasons down per rule.
per_rule = (quarantine_df["reason"].str.split("; ").explode().value_counts()
            .rename_axis("rule").reset_index(name="bookings"))
if len(per_rule) != len(reason_counts):
    print("Per individual rule (a booking with several problems counts under each):")
    display(per_rule)

Lines read:   283
Passed:       259
Quarantined:  24

Quarantined per reason (one row per booking):


,reason,bookings
0,student.id: String should match pattern '^[0-9]{2}-[0-9]{4}$',4
1,total does not equal the sum of qty x price over the tickets,4
2,"tickets.type: Input should be 'standard', 'vip' or 'student'",3
3,tickets.qty: Input should be greater than or equal to 1,3
4,duplicate booking_id,3
5,booked_at is not before the event's date,3
6,event: missing,2
7,"tickets: List should have at least 1 item after validation, not 0",2


**Why the duplicate check runs after validation.** The cell below runs the two steps in the opposite order on the same
file (deduplicate the raw lines first, then validate) and compares the result with the pipeline above.

In [6]:
def run_dedup_first(path):
    """The swapped order: keep the first raw copy of each booking_id, then validate."""
    seen, kept, q = set(), [], Counter()
    for _, doc, parse_error, _ in read_lines(path):
        if parse_error:
            q[parse_error] += 1
            continue
        bid = doc.get("booking_id") if isinstance(doc, dict) else None
        if bid is not None and bid in seen:
            q[DUPLICATE] += 1
            continue
        seen.add(bid)
        try:
            kept.append(Booking.model_validate(doc))
        except ValidationError as err:
            q[reason_from(err)] += 1
    return kept, q


swapped_kept, swapped_q = run_dedup_first(DATA)
ours = Counter(quarantine_df["reason"])
compare = (pd.DataFrame({"validate_then_dedup": ours, "dedup_then_validate": swapped_q})
           .fillna(0).astype(int).sort_index())
compare.loc["(passed)"] = [passed, len(swapped_kept)]
display(compare)

lost = sorted({b.booking_id for b in valid} - {b.booking_id for b in swapped_kept})
same_counts = compare["validate_then_dedup"].equals(compare["dedup_then_validate"])
print("Counts identical in both orders:", same_counts)
print("Valid bookings the swapped order would lose:", lost or "none")

,validate_then_dedup,dedup_then_validate
booked_at is not before the event's date,3,3
duplicate booking_id,3,3
event: missing,2,2
student.id: String should match pattern '^[0-9]{2}-[0-9]{4}$',4,4
tickets.qty: Input should be greater than or equal to 1,3,3
"tickets.type: Input should be 'standard', 'vip' or 'student'",3,3
"tickets: List should have at least 1 item after validation, not 0",2,2
total does not equal the sum of qty x price over the tickets,4,4
(passed),259,259


Counts identical in both orders: True
Valid bookings the swapped order would lose: none


## Part B: Flatten (Lab 6)

### B.1 Two flat tables
* `bookings` – one row per booking; `student` and `event` become prefixed columns, no `tickets` column.
* `booking_tickets` – one row per ticket entry with `booking_id`, `type`, `qty`, `price` and `amount = qty × price`.

In [7]:
valid_dicts = [b.model_dump() for b in valid]

# One row per booking: nested objects become student_* / event_* columns; tickets are left out.
bookings = pd.json_normalize([{k: v for k, v in d.items() if k != "tickets"} for d in valid_dicts], sep="_")
bookings["booked_at"] = pd.to_datetime(bookings["booked_at"], utc=True)
bookings["event_date"] = pd.to_datetime(bookings["event_date"])
first_cols = ["booking_id", "booked_at",
              "student_id", "student_name", "student_faculty",
              "event_id", "event_title", "event_club", "event_date", "event_venue",
              "total", "payment", "status", "promo_code"]
bookings = bookings[first_cols + [c for c in bookings.columns if c not in first_cols]]

# One row per ticket entry.
booking_tickets = pd.json_normalize(valid_dicts, record_path="tickets", meta=["booking_id"])
booking_tickets["amount"] = booking_tickets["qty"] * booking_tickets["price"]
ticket_cols = ["booking_id", "type", "qty", "price", "amount"]
booking_tickets = booking_tickets[ticket_cols + [c for c in booking_tickets.columns if c not in ticket_cols]]

print(f"bookings:        {len(bookings)} rows x {bookings.shape[1]} columns")
print(f"booking_tickets: {len(booking_tickets)} rows x {booking_tickets.shape[1]} columns")
display(bookings.head())
display(booking_tickets.head())

bookings:        259 rows x 14 columns
booking_tickets: 314 rows x 5 columns


,booking_id,booked_at,student_id,student_name,student_faculty,event_id,event_title,event_club,event_date,event_venue,total,payment,status,promo_code
0,B0001,2026-09-28 15:46:00+00:00,60-3778,Nour Mansour,IET,EV-01,Robotics Demo Day,Robotics,2026-10-15,Hall A,340.0,card,cancelled,NaN
1,B0002,2026-09-27 12:29:00+00:00,61-3826,Laila Adel,IET,EV-01,Robotics Demo Day,Robotics,2026-10-15,Hall A,40.0,card,confirmed,NaN
2,B0003,2026-10-13 20:19:00+00:00,58-5324,Ziad Kamel,IET,EV-02,Inter-faculty Debate Final,Debate,2026-10-22,Auditorium,60.0,wallet,confirmed,NaN
3,B0004,2026-11-01 06:42:00+00:00,58-6052,Seif Ibrahim,Management,EV-07,Open Mic Night,Music,2026-11-19,Cafeteria stage,105.0,card,confirmed,CLUB10
4,B0005,2026-10-28 19:06:00+00:00,58-9891,Ali Fouad,Management,EV-06,Line-follower Challenge,Robotics,2026-11-12,Hall A,20.0,cash,confirmed,NaN


,booking_id,type,qty,price,amount
0,B0001,standard,1,40.0,40.0
1,B0001,vip,3,100.0,300.0
2,B0002,standard,1,40.0,40.0
3,B0003,standard,2,30.0,60.0
4,B0004,standard,3,35.0,105.0


### B.2 Checks

In [8]:
# booking_id is unique in bookings (one row per booking)
assert bookings["booking_id"].is_unique, "booking_id is not unique in bookings"
assert len(bookings) == passed

# every booking has tickets, and the amounts of its tickets add up to its total
amount_per_booking = booking_tickets.groupby("booking_id")["amount"].sum()
check = bookings.set_index("booking_id")[["total"]].join(amount_per_booking)
assert check["amount"].notna().all(), "a booking has no ticket rows"
assert ((check["amount"] - check["total"]).abs() < 0.005).all(), "ticket amounts do not add up to total"

assert set(booking_tickets["booking_id"]) == set(bookings["booking_id"])
print("All checks passed: booking_id unique; ticket amounts add up to total for all",
      len(check), "bookings.")

All checks passed: booking_id unique; ticket amounts add up to total for all 259 bookings.


### B.3 Schema drift
Every key path of every document read (valid or not) is compared with the documented example. Any key that the example
does not have is drift; *from when* is the earliest `booked_at` of a document carrying it.

In [9]:
DOCUMENTED = {
    "booking_id", "booked_at", "total", "payment", "status",
    "student", "student.id", "student.name", "student.faculty",
    "event", "event.id", "event.title", "event.club", "event.date", "event.venue",
    "tickets", "tickets[].type", "tickets[].qty", "tickets[].price",
}


def key_paths(obj, prefix=""):
    """All key paths in a nested document, e.g. 'student.id', 'tickets[].qty'."""
    if isinstance(obj, dict):
        for k, v in obj.items():
            path = f"{prefix}{k}"
            yield path
            yield from key_paths(v, path + ".")
    elif isinstance(obj, list):
        for item in obj:
            yield from key_paths(item, prefix.rstrip(".") + "[].")


def booked_ts(doc):
    """booked_at of a raw document as a UTC timestamp (NaT if it is missing or unreadable)."""
    try:
        ts = pd.Timestamp(doc.get("booked_at"))
        return ts.tz_localize("UTC") if ts.tzinfo is None else ts.tz_convert("UTC")
    except (ValueError, TypeError):
        return pd.NaT


drift_rows = []
for line_no, doc in docs:
    if not isinstance(doc, dict):
        continue
    for path in sorted(set(key_paths(doc)) - DOCUMENTED):
        drift_rows.append({"field": path, "line": line_no, "booking_id": doc.get("booking_id"),
                           "booked_at": booked_ts(doc)})
drift = pd.DataFrame(drift_rows, columns=["field", "line", "booking_id", "booked_at"])

if drift.empty:
    print("No drift: every document matches the documented example.")
else:
    summary = (drift.groupby("field")
               .agg(documents=("line", "count"), first_booked_at=("booked_at", "min"),
                    first_line_in_file=("line", "min"))
               .reset_index())
    display(summary)

    all_docs = pd.DataFrame([{"line": n, "booking_id": d.get("booking_id"), "booked_at": booked_ts(d)}
                             for n, d in docs if isinstance(d, dict)])
    quarantined_lines = set(quarantine_df["line"])
    for field, first in zip(summary["field"], summary["first_booked_at"]):
        has = all_docs["line"].isin(drift.loc[drift["field"] == field, "line"])
        before, after = all_docs["booked_at"] < first, all_docs["booked_at"] >= first
        print(f"\n'{field}' first appears in a document booked at {first} (UTC), i.e. from {first.date()}.")
        print(f"  booked before {first.date()}: {before.sum():>3} documents, {(has & before).sum()} with '{field}'")
        print(f"  booked from   {first.date()}: {after.sum():>3} documents, {(has & after).sum()} with '{field}'")
        missing_after = all_docs[after & ~has].assign(quarantined=lambda d: d["line"].isin(quarantined_lines))
        if len(missing_after):
            print(f"  documents from {first.date()} on without it:")
            display(missing_after)
        if "." not in field and "[]" not in field:  # value breakdown for top-level fields
            values = pd.Series([d.get(field) for _, d in docs if isinstance(d, dict) and field in d])
            print(f"  values of '{field}' (None = sent as null):", values.value_counts(dropna=False).to_dict())

,field,documents,first_booked_at,first_line_in_file
0,promo_code,193,2026-10-20 01:05:00+00:00,4



'promo_code' first appears in a document booked at 2026-10-20 01:05:00+00:00 (UTC), i.e. from 2026-10-20.
  booked before 2026-10-20:  89 documents, 0 with 'promo_code'
  booked from   2026-10-20: 194 documents, 193 with 'promo_code'
  documents from 2026-10-20 on without it:


,line,booking_id,booked_at,quarantined
169,170,B0168,2026-10-31 10:00:00+00:00,True


  values of 'promo_code' (None = sent as null): {nan: 97, 'CLUB10': 50, 'EARLY': 46}


In [10]:
# What the pipeline does with documents that lack promo_code: the model's default (None) is used,
# so bookings.promo_code is NULL for them; documents that send promo_code: null look the same.
pd.DataFrame({
    "bookings": [len(bookings)],
    "promo_code NULL": [bookings["promo_code"].isna().sum()],
    "promo_code set": [bookings["promo_code"].notna().sum()],
})

,bookings,promo_code NULL,promo_code set
0,259,172,87


## Part C: Model and store (Lab 5)

### C.1 Star schema in DuckDB
Grain of the fact: **one row per ticket entry** (a row of `booking_tickets`).

* `dim_event` – one row per `event_id`, surrogate key `event_key`
* `dim_student` – one row per `student_id`, surrogate key `student_key`
* `fact_ticket_sales` – `booking_id, event_key, student_key, ticket_type, qty, price, amount, status, booked_date_key` (`YYYYMMDD`, from `booked_at` in UTC)

If the same natural key ever arrives with different attributes, the dimension keeps the attributes of the most recent booking
(`arg_max(..., booked_at)`), so each dimension still has exactly one row per natural key.

In [11]:
con = duckdb.connect()                 # in-memory database
con.execute("SET TimeZone = 'UTC'")    # booked_at is UTC; date keys and months must not shift
con.register("bookings_df", bookings)
con.register("booking_tickets_df", booking_tickets)
con.execute("CREATE OR REPLACE TABLE bookings AS SELECT * FROM bookings_df")
con.execute("CREATE OR REPLACE TABLE booking_tickets AS SELECT * FROM booking_tickets_df")

con.execute("""
CREATE OR REPLACE TABLE dim_event AS
SELECT CAST(ROW_NUMBER() OVER (ORDER BY event_id) AS INTEGER) AS event_key,
       event_id, title, club, event_date, venue
FROM (
    SELECT event_id,
           arg_max(event_title, booked_at)               AS title,
           arg_max(event_club,  booked_at)               AS club,
           CAST(arg_max(event_date, booked_at) AS DATE)  AS event_date,
           arg_max(event_venue, booked_at)               AS venue
    FROM bookings
    GROUP BY event_id
)
""")

con.execute("""
CREATE OR REPLACE TABLE dim_student AS
SELECT CAST(ROW_NUMBER() OVER (ORDER BY student_id) AS INTEGER) AS student_key,
       student_id, name, faculty
FROM (
    SELECT student_id,
           arg_max(student_name,    booked_at) AS name,
           arg_max(student_faculty, booked_at) AS faculty
    FROM bookings
    GROUP BY student_id
)
""")

con.execute("""
CREATE OR REPLACE TABLE fact_ticket_sales AS
SELECT t.booking_id,
       e.event_key,
       s.student_key,
       t.type   AS ticket_type,
       t.qty,
       t.price,
       t.amount,
       b.status,
       CAST(strftime(b.booked_at, '%Y%m%d') AS INTEGER) AS booked_date_key
FROM booking_tickets t
JOIN bookings    b ON b.booking_id = t.booking_id
JOIN dim_event   e ON e.event_id   = b.event_id
JOIN dim_student s ON s.student_id = b.student_id
ORDER BY t.booking_id
""")

for table in ["dim_event", "dim_student", "fact_ticket_sales"]:
    n = con.execute(f"SELECT COUNT(*) FROM {table}").fetchall()[0][0]
    print(f"{table}: {n} rows")
display(con.sql("SELECT * FROM dim_event").df())
display(con.sql("SELECT * FROM dim_student LIMIT 5").df())
display(con.sql("SELECT * FROM fact_ticket_sales LIMIT 5").df())

dim_event: 8 rows
dim_student: 103 rows
fact_ticket_sales: 314 rows


,event_key,event_id,title,club,event_date,venue
0,1,EV-01,Robotics Demo Day,Robotics,2026-10-15,Hall A
1,2,EV-02,Inter-faculty Debate Final,Debate,2026-10-22,Auditorium
2,3,EV-03,Autumn Concert,Music,2026-10-29,Open-air theatre
3,4,EV-04,Five-a-side Cup,Football,2026-11-05,Sports field
4,5,EV-05,Blitz Chess Open,Chess,2026-11-07,C7.201
5,6,EV-06,Line-follower Challenge,Robotics,2026-11-12,Hall A
6,7,EV-07,Open Mic Night,Music,2026-11-19,Cafeteria stage
7,8,EV-08,Model UN Debate,Debate,2026-11-26,Auditorium


,student_key,student_id,name,faculty
0,1,58-1324,Nour Fouad,MET
1,2,58-1449,Farida Mansour,EMS
2,3,58-1801,Karim Kamel,IET
3,4,58-2165,Farida Hassan,Management
4,5,58-2307,Farida Hassan,Pharmacy


,booking_id,event_key,student_key,ticket_type,qty,price,amount,status,booked_date_key
0,B0001,1,61,standard,1,40.0,40.0,cancelled,20260928
1,B0001,1,61,vip,3,100.0,300.0,cancelled,20260928
2,B0002,1,84,standard,1,40.0,40.0,confirmed,20260927
3,B0003,2,14,standard,2,30.0,60.0,confirmed,20261013
4,B0004,7,15,standard,3,35.0,105.0,confirmed,20261101


### C.2 Checks

In [12]:
fact_rows, fact_amount = con.execute("SELECT COUNT(*), SUM(amount) FROM fact_ticket_sales").fetchall()[0]
bookings_total = con.execute("SELECT SUM(total) FROM bookings").fetchall()[0][0]

# the fact has as many rows as booking_tickets (the grain is preserved by the joins)
assert fact_rows == len(booking_tickets), f"{fact_rows} fact rows vs {len(booking_tickets)} ticket rows"
# the sum of amount in the fact equals the sum of total in bookings
assert math.isclose(fact_amount, bookings_total, abs_tol=0.005), f"{fact_amount} vs {bookings_total}"

# surrogate keys: unique per dimension, one row per natural key, and every fact row finds its dimensions
assert con.execute("SELECT COUNT(*) = COUNT(DISTINCT event_key) AND COUNT(*) = COUNT(DISTINCT event_id) FROM dim_event").fetchall()[0][0]
assert con.execute("SELECT COUNT(*) = COUNT(DISTINCT student_key) AND COUNT(*) = COUNT(DISTINCT student_id) FROM dim_student").fetchall()[0][0]
assert con.execute("SELECT COUNT(*) FROM fact_ticket_sales WHERE event_key IS NULL OR student_key IS NULL").fetchall()[0][0] == 0

print(f"Fact rows = booking_tickets rows = {fact_rows}")
print(f"SUM(amount) in fact = SUM(total) in bookings = {fact_amount:,.2f} EGP")

Fact rows = booking_tickets rows = 314
SUM(amount) in fact = SUM(total) in bookings = 31,285.00 EGP


In [13]:
# Why total is not a fact column: at ticket grain it would repeat once per ticket entry of a booking.
wrong = con.execute("""
    SELECT SUM(b.total) FROM fact_ticket_sales f JOIN bookings b USING (booking_id)
""").fetchall()[0][0]
multi = con.execute("""
    SELECT COUNT(*) FROM (SELECT booking_id FROM fact_ticket_sales GROUP BY booking_id HAVING COUNT(*) > 1)
""").fetchall()[0][0]
print(f"SUM(total) if total sat on every fact row: {wrong:,.2f} EGP")
print(f"Real revenue (SUM(amount) = SUM(total) over bookings): {bookings_total:,.2f} EGP")
print(f"Overstated by {wrong - bookings_total:,.2f} EGP, because {multi} bookings have more than one ticket entry.")

SUM(total) if total sat on every fact row: 42,000.00 EGP
Real revenue (SUM(amount) = SUM(total) over bookings): 31,285.00 EGP
Overstated by 10,715.00 EGP, because 55 bookings have more than one ticket entry.


### C.3 Confirmed bookings: tickets sold and revenue per club and ticket type (from the star)

In [14]:
answer = con.sql("""
    SELECT e.club,
           f.ticket_type,
           CAST(SUM(f.qty) AS BIGINT) AS tickets_sold,
           SUM(f.amount) AS revenue_egp
    FROM fact_ticket_sales f
    JOIN dim_event e ON e.event_key = f.event_key
    WHERE f.status = 'confirmed'
    GROUP BY e.club, f.ticket_type
    ORDER BY e.club, f.ticket_type
""").df()
display(answer)
print(f"Total confirmed: {answer['tickets_sold'].sum()} tickets, {answer['revenue_egp'].sum():,.2f} EGP")

,club,ticket_type,tickets_sold,revenue_egp
0,Chess,standard,45,900.0
1,Chess,student,44,440.0
2,Debate,standard,65,1950.0
3,Debate,student,34,510.0
4,Debate,vip,40,3000.0
5,Football,standard,64,1600.0
6,Music,standard,67,2930.0
7,Music,student,20,1000.0
8,Music,vip,78,9990.0
9,Robotics,standard,60,2400.0


Total confirmed: 577 tickets, 28,320.00 EGP


### C.4 Save as Parquet in `star/`
Each dimension is one file; the fact is partitioned by the month the booking was made (`booked_month=YYYY-MM` folders).
The folder is rebuilt from scratch on every run.

In [15]:
if STAR_DIR.exists():
    shutil.rmtree(STAR_DIR)
STAR_DIR.mkdir()

star = STAR_DIR.as_posix()
con.execute(f"COPY dim_event   TO '{star}/dim_event.parquet'   (FORMAT PARQUET)")
con.execute(f"COPY dim_student TO '{star}/dim_student.parquet' (FORMAT PARQUET)")
con.execute(f"""
COPY (
    SELECT *, strftime(strptime(CAST(booked_date_key AS VARCHAR), '%Y%m%d'), '%Y-%m') AS booked_month
    FROM fact_ticket_sales
) TO '{star}/fact_ticket_sales' (FORMAT PARQUET, PARTITION_BY (booked_month))
""")

for p in sorted(STAR_DIR.rglob("*.parquet")):
    print(p.as_posix())

star/dim_event.parquet
star/dim_student.parquet
star/fact_ticket_sales/booked_month=2026-09/data_0.parquet
star/fact_ticket_sales/booked_month=2026-10/data_0.parquet
star/fact_ticket_sales/booked_month=2026-11/data_0.parquet


In [16]:
# Read the files back and check they hold the whole star.
back = con.sql(f"""
    SELECT booked_month, COUNT(*) AS fact_rows, SUM(amount) AS amount
    FROM read_parquet('{star}/fact_ticket_sales/*/*.parquet', hive_partitioning = true)
    GROUP BY booked_month ORDER BY booked_month
""").df()
display(back)
assert back["fact_rows"].sum() == fact_rows
assert math.isclose(back["amount"].sum(), fact_amount, abs_tol=0.005)
assert con.execute(f"SELECT COUNT(*) FROM '{star}/dim_event.parquet'").fetchall()[0][0] == \
       con.execute("SELECT COUNT(*) FROM dim_event").fetchall()[0][0]
assert con.execute(f"SELECT COUNT(*) FROM '{star}/dim_student.parquet'").fetchall()[0][0] == \
       con.execute("SELECT COUNT(*) FROM dim_student").fetchall()[0][0]
print("Parquet files read back and match the star.")

,booked_month,fact_rows,amount
0,2026-09,12,2160.0
1,2026-10,179,17050.0
2,2026-11,123,12075.0


Parquet files read back and match the star.


## Part D: Numbers for the paragraph
Everything the README paragraph quotes, computed from this run.

In [17]:
drift_first = drift.groupby("field")["booked_at"].min().to_dict() if not drift.empty else {}
print(f"Lines read: {lines_read} | passed (bookings left): {passed} | quarantined: {quarantined}")
for reason, n in quarantine_df["reason"].value_counts().items():
    print(f"  {n:>3}  {reason}")
print(f"Duplicate-check order changes the counts on this file: {not same_counts}")
print("Drifted field(s):", {f: str(t) for f, t in drift_first.items()} or "none")
print(f"Ticket entries (fact rows): {fact_rows} | revenue SUM(amount): {fact_amount:,.2f} EGP "
      f"| SUM(total) over the fact would be: {wrong:,.2f} EGP")

Lines read: 283 | passed (bookings left): 259 | quarantined: 24
    4  student.id: String should match pattern '^[0-9]{2}-[0-9]{4}$'
    4  total does not equal the sum of qty x price over the tickets
    3  tickets.type: Input should be 'standard', 'vip' or 'student'
    3  tickets.qty: Input should be greater than or equal to 1
    3  duplicate booking_id
    3  booked_at is not before the event's date
    2  event: missing
    2  tickets: List should have at least 1 item after validation, not 0
Duplicate-check order changes the counts on this file: False
Drifted field(s): {'promo_code': '2026-10-20 01:05:00+00:00'}
Ticket entries (fact rows): 314 | revenue SUM(amount): 31,285.00 EGP | SUM(total) over the fact would be: 42,000.00 EGP
